In [1]:
!pip install deep-sort-realtime filterpy

Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110630 sha256=2691c9e549b0818906fe834b5f4661c2735537de20fc663d8d96e6aec86bde49
  Stored in directory: c:\users\mehdi\appdata\local\pip\cache\wheels\79\33\43\53b597b8f63de80842202a5fed633eea6f5ce3e3f6c6efbab8
Successfully built filterpy


In [15]:
import cv2
import numpy as np
import pandas as pd
from collections import defaultdict
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from filterpy.kalman import KalmanFilter

In [16]:
# Load YOLO model
model = YOLO(r"..\..\detection\fine_tunning_3\runs\detect\train\weights\best.pt")

In [17]:
# ---------------- Player tracker (DeepSORT) ----------------
tracker = DeepSort(
    max_age=50,
    n_init=1,
    max_iou_distance=0.9,
    max_cosine_distance=0.6,
    nn_budget=100,
    bgr=True
)


# ---------------- Ball tracker (Kalman filter, single object) ----------------
class BallTracker:
    """
    Tracks a single ball with a constant-velocity Kalman filter.
    Not DeepSORT: the ball is one object, so re-identification adds nothing
    and DeepSORT tends to spawn duplicate IDs when a small, fast-moving
    object briefly fails IOU/appearance matching.
    """

    def __init__(self, max_missed=30, max_jump_dist=150):
        self.kf = KalmanFilter(dim_x=4, dim_z=2)  # state: [x, y, vx, vy]
        self.kf.F = np.array([[1, 0, 1, 0],
                               [0, 1, 0, 1],
                               [0, 0, 1, 0],
                               [0, 0, 0, 1]])
        self.kf.H = np.array([[1, 0, 0, 0],
                               [0, 1, 0, 0]])
        self.kf.P *= 100
        self.kf.R *= 5      # measurement noise (YOLO detection jitter)
        self.kf.Q *= 0.1    # process noise (ball motion)

        self.initialized = False
        self.missed_frames = 0
        self.max_missed = max_missed          # frames tolerated with no detection
        self.max_jump_dist = max_jump_dist    # px, plausible frame-to-frame movement

    def _select_best_detection(self, detections):
        """detections: list of (cx, cy, conf). Returns (cx, cy) or None."""
        if not detections:
            return None

        if not self.initialized:
            # no trajectory yet -> trust YOLO confidence
            return max(detections, key=lambda d: d[2])[:2]

        predicted_x, predicted_y = self.kf.x[0], self.kf.x[1]
        plausible = [
            d for d in detections
            if np.hypot(d[0] - predicted_x, d[1] - predicted_y) < self.max_jump_dist
        ]
        candidates = plausible if plausible else detections
        best = min(candidates, key=lambda d: np.hypot(d[0] - predicted_x, d[1] - predicted_y))
        return best[:2]

    def update(self, detections):
        """detections: list of (cx, cy, conf) for this frame. Returns (x, y) or None."""
        detection = self._select_best_detection(detections)

        if detection is not None:
            z = np.array(detection)
            if not self.initialized:
                self.kf.x = np.array([z[0], z[1], 0, 0])
                self.initialized = True
            else:
                self.kf.predict()
                self.kf.update(z)
            self.missed_frames = 0
        else:
            if self.initialized:
                self.kf.predict()
                self.missed_frames += 1
                if self.missed_frames > self.max_missed:
                    self.initialized = False

        if self.initialized:
            return int(self.kf.x[0]), int(self.kf.x[1])
        return None


ball_tracker = BallTracker()

In [18]:
# Open video
cap = cv2.VideoCapture("../../../data/videos/video1.mp4")

fps = int(cap.get(cv2.CAP_PROP_FPS))
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    "../../../outputs/videos/tracking_result2.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (frame_width, frame_height)
)

In [19]:
def get_dominant_jersey_color(crop):
    if crop is None or crop.size == 0:
        return None
    h, w = crop.shape[:2]
    if h < 10 or w < 10:
        return None

    y1, y2 = int(h * 0.15), int(h * 0.45)
    x1, x2 = int(w * 0.25), int(w * 0.75)
    torso = crop[y1:y2, x1:x2]
    if torso.size == 0:
        return None

    hsv = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)

    # mask grass
    green_mask = cv2.inRange(hsv, (35, 40, 40), (85, 255, 255))
    # mask skin tones (rough range, tune to your footage)
    skin_mask = cv2.inRange(hsv, (0, 20, 70), (20, 150, 255))
    # mask near-white (socks, lines, overexposed highlights)
    white_mask = cv2.inRange(hsv, (0, 0, 200), (180, 40, 255))
    # mask near-black (shadows)
    black_mask = cv2.inRange(hsv, (0, 0, 0), (180, 255, 40))

    exclude = cv2.bitwise_or(green_mask, skin_mask)
    exclude = cv2.bitwise_or(exclude, white_mask)
    exclude = cv2.bitwise_or(exclude, black_mask)
    valid_mask = cv2.bitwise_not(exclude)

    lab = cv2.cvtColor(torso, cv2.COLOR_BGR2LAB)
    pixels = lab.reshape(-1, 3)
    mask_flat = valid_mask.reshape(-1) > 0

    if mask_flat.sum() < 20:  # not enough clean jersey pixels
        return None

    valid_pixels = pixels[mask_flat]
    # median is more robust to outliers than mean
    dominant = np.median(valid_pixels[:, 1:3], axis=0)  # only A, B (ignore L)
    return dominant  # 2-value vector, not 256

In [20]:
classifier = "null"
player_samples = defaultdict(list)   # track_id -> list of [A, B] samples
player_features = {}                 # track_id -> final averaged [A, B] feature
team_memory = {}                     # track_id -> "Team A" / "Team B"
clustering_done = False
SAMPLES_PER_TRACK = 5                # crops to average before locking a player's color
MIN_PLAYERS_FOR_CLUSTERING = 14# wait for a diverse enough sample before first clustering
tracking_records = []   # will become the DataFrame

time = 0
while cap.isOpened():
    time += 1
    ret, frame = cap.read()
    if not ret:
        break

    # ---------------- YOLO ----------------
    results = model(frame, conf=0.3, iou=0.5)

    player_detections = []   # goes to DeepSORT
    ball_candidates = []     # goes to the Kalman ball tracker, (cx, cy, conf)

    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = float(box.conf[0])
            cls = int(box.cls[0])
            class_name = model.names[cls]

            if class_name == "Ball":
                cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                ball_candidates.append((cx, cy, conf))
            else:
                w, h = x2 - x1, y2 - y1
                player_detections.append(([x1, y1, w, h], conf, cls))

    # ---------------- DeepSORT (players / keeper / refs) ----------------
    tracks = tracker.update_tracks(player_detections, frame=frame)

    # ---------------- Ball (single object, Kalman) ----------------
    ball_pos = ball_tracker.update(ball_candidates)
    if ball_pos is not None:
        cv2.circle(frame, ball_pos, 8, (0, 0, 255), -1)

    # ---------------- Player tracking / drawing ----------------
    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        class_id = track.get_det_class()
        class_name = model.names[class_id]

        l, t, r, b = map(int, track.to_ltrb())
        l = max(0, l)
        t = max(0, t)
        r = min(frame.shape[1], r)
        b = min(frame.shape[0], b)


        # ---------------- Log record for CSV ----------------
        team_label = team_memory.get(track_id, None)  # "Team A" / "Team B" / None if not yet resolved
        confidence = track.get_det_conf()              # can be None on predicted (non-detected) frames

        tracking_records.append({
            "frame": time,
            "id": track_id,
            "class_id": class_id,
            "class_name": class_name,
            "team": team_label,
            "confidence": confidence,
            "x1": l,
            "y1": t,
            "x2": r,
            "y2": b,
        })

        # ---------------- Player Feature Extraction ----------------
        if class_name == "Player" and track_id not in player_features:
            if len(player_samples[track_id]) < SAMPLES_PER_TRACK:
                player_crop = frame[t:b, l:r]
                color = get_dominant_jersey_color(player_crop)
                if color is not None:
                    player_samples[track_id].append(color)

            if len(player_samples[track_id]) >= SAMPLES_PER_TRACK:
                player_features[track_id] = np.median(player_samples[track_id], axis=0)

        # ---------------- Draw ----------------
        if track_id in team_memory:
            label = f"{team_memory[track_id]} | ID:{track_id}"
        else:
            label = f"{class_name} | ID:{track_id}"

        cv2.rectangle(frame, (l, t), (r, b), (0, 255, 0), 2)
        cv2.putText(frame, label, (l, t - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # ---------------- Team classification & clustering ----------------
    # Cluster ONCE to fix which KMeans label ("0"/"1") means which team, then only
    # ever use classifier.predict() afterwards. Re-running fit_predict() on every new
    # player is what was causing teams to get swapped a few seconds in: KMeans gives
    # no guarantee that cluster "0" means the same team across separate fits.
    if not clustering_done and len(player_features) >= MIN_PLAYERS_FOR_CLUSTERING:
        ids = list(player_features.keys())
        features = np.array(list(player_features.values()))

        kmeans = KMeans(n_clusters=2, random_state=42, n_init="auto")
        labels = kmeans.fit_predict(features)

        classifier = RandomForestClassifier(n_estimators=100, random_state=42)
        classifier.fit(features, labels)

        for track_id, cluster in zip(ids, labels):
            team_memory[track_id] = "Team A" if cluster == 0 else "Team B"

        clustering_done = True

    elif clustering_done and not isinstance(classifier, str):
        # Newly resolved players: predict with the fixed classifier, never re-cluster.
        new_ids = [tid for tid in player_features if tid not in team_memory]
        if new_ids:
            new_features = np.array([player_features[tid] for tid in new_ids])
            new_labels = classifier.predict(new_features)
            for track_id, cluster in zip(new_ids, new_labels):
                team_memory[track_id] = "Team A" if cluster == 0 else "Team B"

    out.write(frame)
    cv2.imshow("Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        tracking_df = pd.DataFrame(tracking_records)
        tracking_df.to_csv("../../../outputs/csv_data/tracking_data.csv", index=False)
        print(f"Saved {len(tracking_df)} rows to tracking_data.csv")
        break

cap.release()
out.release()
cv2.destroyAllWindows()


0: 384x640 1 Keeper, 18 Players, 2 Refs, 146.8ms
Speed: 4.9ms preprocess, 146.8ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Ball, 1 Keeper, 18 Players, 2 Refs, 85.2ms
Speed: 2.8ms preprocess, 85.2ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Ball, 1 Keeper, 19 Players, 2 Refs, 63.5ms
Speed: 2.4ms preprocess, 63.5ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Ball, 1 Keeper, 19 Players, 1 Ref, 61.1ms
Speed: 3.4ms preprocess, 61.1ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Ball, 1 Keeper, 19 Players, 1 Ref, 57.2ms
Speed: 2.4ms preprocess, 57.2ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Ball, 1 Keeper, 19 Players, 1 Ref, 59.4ms
Speed: 2.5ms preprocess, 59.4ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Ball, 1 Keeper, 19 Players, 1 Ref, 59.2ms
Speed: 2.0ms preprocess, 59.2